In [1]:
import hashlib, os, platform, uuid

# 修改为自己的学号
STUDENT_ID = 20244080210

CONDA_ENV = os.environ.get("CONDA_DEFAULT_ENV", "base")
print(f"[CONDA 环境] {CONDA_ENV}")

MACHINE_FP = hashlib.sha256(
    f"{platform.node()}{platform.machine()}{platform.system()}{uuid.getnode()}".encode()
).hexdigest()[:16]

RUN_ID = f"{STUDENT_ID}-{MACHINE_FP}"
print(f"[学号] {STUDENT_ID}")
print(f"[机器指纹] {MACHINE_FP}")
print(f"[RUN_ID] {RUN_ID}")

[CONDA 环境] exp01
[学号] 20244080210
[机器指纹] 75bd2a33be05d306
[RUN_ID] 20244080210-75bd2a33be05d306


## 字符串的乐趣

In [2]:
def is_palindrome(s):
    # 去掉空格并全部转为小写
    cleaned = s.replace(" ", "").lower()
    return cleaned == cleaned[::-1]

def is_abecedarian(s):
    # 去掉空格
    s = s.replace(" ", "")
    for i in range(len(s) - 1):
        # 如果前一个字母大于后一个字母，说明不是字母顺序的
        if s[i] > s[i + 1]:
            return False
    return True

def remove_vowels(s):
    vowels = "aeiou"
    # 提取不是元音的字符并拼接
    return "".join(ch for ch in s if ch not in vowels)

# --- 测试代码 ---
print("测试回文:")
print(is_palindrome("taco cat"))    # True
print(is_palindrome("hello"))       # False

print("\n测试字母顺序:")
print(is_abecedarian("adder"))      # True
print(is_abecedarian("dog"))        # False

print("\n测试去元音:")
print(remove_vowels("audio"))       # d

测试回文:
True
False

测试字母顺序:
True
False

测试去元音:
d


## 列表的乐趣

In [3]:
def list_reverse(lst):
    if not isinstance(lst, list):
        raise TypeError("输入必须是列表 (list)")
    return lst[::-1]

def is_sorted(seq):
    # 只遍历一次，比较相邻元素
    for i in range(len(seq) - 1):
        if seq[i] > seq[i + 1]:
            return False
    return True

# --- 测试代码 ---
print("测试列表反转:")
print(list_reverse([1, 2, 3]))      # [3, 2, 1]

print("\n测试排序:")
print(is_sorted([1, 2, 3, 5]))      # True
print(is_sorted([1, 3, 2]))         # False

测试列表反转:
[3, 2, 1]

测试排序:
True
False


## 作为向量的元组

In [4]:
def vec_scalar_mult(t, s):
    if not isinstance(t, tuple):
        raise TypeError("第一个参数必须是元组")
    if not isinstance(s, (int, float)):
        raise TypeError("标量必须是数字")
    return tuple(x * s for x in t)

def vec_inner_product(x, y):
    if not isinstance(x, tuple) or not isinstance(y, tuple):
        raise TypeError("两个参数都必须是元组")
    if len(x) != len(y):
        raise ValueError("两个向量长度必须一致")
    return float(sum(a * b for a, b in zip(x, y)))

def check_valid_mx(m):
    if not isinstance(m, tuple):
        return False
    for row in m:
        if not isinstance(row, tuple):
            return False
    if len(m) == 0:
        return True
    n = len(m[0])
    for row in m:
        if len(row) != n:
            return False
        for item in row:
            if not isinstance(item, (int, float)):
                return False
    return True

def mx_vec_mult(mx, v):
    if not check_valid_mx(mx) or not isinstance(v, tuple):
        raise TypeError("输入矩阵或向量格式不合法")
    if len(mx) == 0:
        return ()
    if len(mx[0]) != len(v):
        raise ValueError("矩阵列数必须等于向量长度")
    
    return tuple(
        sum(row[j] * v[j] for j in range(len(v)))
        for row in mx
    )

# --- 测试代码 ---
print("测试标量乘法:", vec_scalar_mult((1, 2, 3), 2))    # (2, 4, 6)
print("测试内积:", vec_inner_product((1, 2), (3, 4)))      # 11.0

mx = ((1, 2), (3, 4))
v = (1, 1)
print("矩阵乘向量:", mx_vec_mult(mx, v))                   # (3, 7)

测试标量乘法: (2, 4, 6)
测试内积: 11.0
矩阵乘向量: (3, 7)


## 更多向量的乐趣（面向对象）
**Q2 解答**：维度和元素应该是 **实例属性**。因为每个 Vector 对象都有自己独立的维度大小和坐标数据，如果设为类属性，所有实例将共享同一份数据，这违背了面向对象的原则。

In [5]:
class Vector:
    def __init__(self, dim, elems=None):
        if not isinstance(dim, int) or dim < 0:
            raise TypeError("维度必须是非负整数")
        
        if elems is None:
            elems = [0] * dim
            
        if len(elems) != dim:
            raise ValueError("元素个数必须与维度一致")
            
        self.dim = dim
        self.elems = tuple(elems)  # 用元组保存，防止外部直接修改
        
    def _check_dim(self, other):
        if not isinstance(other, Vector):
            raise TypeError("比较对象必须是 Vector 类型")
        if self.dim != other.dim:
            raise ValueError("向量维度不同，无法比较")

    def __repr__(self):
        return f"Vector({self.dim}, {self.elems})"

    # 比较运算符
    def __eq__(self, other):
        self._check_dim(other)
        return self.elems == other.elems
        
    def __lt__(self, other):
        self._check_dim(other)
        return self.elems < other.elems
        
    def __le__(self, other):
        self._check_dim(other)
        return self.elems <= other.elems
        
    def __gt__(self, other):
        self._check_dim(other)
        return self.elems > other.elems
        
    def __ge__(self, other):
        self._check_dim(other)
        return self.elems >= other.elems

    # 点积方法
    def dot(self, other):
        self._check_dim(other)
        return sum(a * b for a, b in zip(self.elems, other.elems))

    # 乘法运算符
    def __mul__(self, other):
        # 标量乘法
        if isinstance(other, (int, float)):
            return Vector(self.dim, [x * other for x in self.elems])
        # 向量对应位置相乘
        elif isinstance(other, Vector):
            self._check_dim(other)
            return Vector(self.dim, [a * b for a, b in zip(self.elems, other.elems)])
        else:
            return NotImplemented

    # 支持 2 * v 的右乘
    def __rmul__(self, other):
        return self.__mul__(other)

# --- 测试代码 ---
v1 = Vector(2, (2, 4))
v2 = Vector(2, (2, 5))
v3 = Vector(2, (2, 3))
v4 = Vector(2, (4, 5))

print("v1 < v2:", v1 < v2)         # True (先比第一位2==2, 再比第二位4<5)
print("v3 点乘 v4:", v3.dot(v4))   # 23 (2*4 + 3*5)
print("2 * v3:", 2 * v3)           # Vector(2, (4, 6))
print("v3 * v4:", v3 * v4)         # Vector(2, (8, 15))

v1 < v2: True
v3 点乘 v4: 23
2 * v3: Vector(2, (4, 6))
v3 * v4: Vector(2, (8, 15))
